# Compare your reranker against a real stack

In a few minutes and with no API key, this notebook writes a reranker as one Python function, runs it inside a complete RAG pipeline against a strong built-in reranker on a real benchmark, and tests whether the difference is real.

[Lasagna](https://github.com/dolev31/lasagna) swaps one stage of an eight-stage pipeline and holds the other seven fixed. After the run it checks, from what actually reached each stage, that the arms differed only in the stages they named, so the difference you measure belongs to your technique.

The free CPU runtime is enough. For a faster run, choose a GPU before you start: **Runtime > Change runtime type > T4 GPU**. The embedder and the cross-encoder use it on their own.

## 1. Install

The first line installs Lasagna with its evaluation extra, which brings the benchmark loaders. The second fetches nltk's sentence tokenizer data; without it Lasagna splits sentences with a regular expression and records that fallback in the run's bundle. If Colab offers to restart the session afterward, you can decline: every later step runs `lasagna` as a new process.

In [ ]:
import os

# A test of this notebook from a source checkout sets LASAGNA_NOTEBOOK_TEST=1.
if not os.environ.get("LASAGNA_NOTEBOOK_TEST"):
    # For the newest code rather than the PyPI release, run this line instead:
    # %pip install -q "lasagna-rag[eval] @ git+https://github.com/dolev31/lasagna"
    %pip install -q "lasagna-rag[eval]==0.1.0"
    !python -c "import nltk; nltk.download('punkt_tab', quiet=True)"

## 2. Write your technique as one function

A reranker can be a function that scores one passage for a query. This one is keyword overlap: the share of the query's words that appear in the passage. The decorator wraps it in the reranker interface, so it can stand wherever a built-in reranker can. Saved as a file, it can be named in a sweep as `my_ideas:keyword_overlap`, with no package to build.

In [ ]:
%%writefile my_ideas.py
import re

import lasagna


@lasagna.reranker
def keyword_overlap(query, text):
    """Score a passage by the share of the query's words it contains."""
    words = set(re.findall(r"\w+", query.lower()))
    return len(words & set(re.findall(r"\w+", text.lower()))) / max(len(words), 1)

Check that it meets the reranker contract before spending any compute: the right return types, `top_k` respected, empty input handled, and its input left unchanged.

In [ ]:
!lasagna component validate my_ideas:keyword_overlap

## 3. Measure it against a real control in a complete pipeline

The sweep runs three arms on the same 100 SciFact queries, searching the full corpus of 5,183 scientific abstracts. Each arm starts from the `speed` profile (dense retrieval with `all-MiniLM-L6-v2`) and is one change away from the arm before it:

- `cross_encoder` is the control: the built-in cross-encoder reranker, `ms-marco-MiniLM-L-6-v2`.
- `keyword_overlap` puts your function in the reranker slot and changes nothing else.
- `keyword_overlap_bm25` keeps your reranker and swaps the retriever for the built-in BM25. Any of the eight stages swaps the same way.

Every arm also names the same chunking, passages of up to 2,000 characters, so chunking is held fixed. That keeps six in seven abstracts in one piece and makes about 6,000 passages instead of the profile default's 26,000, so embedding and search stay quick on a CPU. One run per arm is enough, because retrieval with fixed models ranks the same way every time.

In [ ]:
%%writefile sweep.yaml
name: does keyword_overlap help
benchmarks:
  - name: beir/scifact
    n_samples: 100
tracks: [retrieval]
output_dir: results
n_runs: 1
pipeline:
  base_profile: speed
  config_overrides:
    - {name: cross_encoder, reranker: cross_encoder, chunker: fixed, chunk_size: 2000}
    - {name: keyword_overlap, reranker: "my_ideas:keyword_overlap", chunker: fixed, chunk_size: 2000}
    - {name: keyword_overlap_bm25, reranker: "my_ideas:keyword_overlap", retriever: bm25, chunker: fixed, chunk_size: 2000}

Run it. The first run downloads SciFact (about 8 MB) and two models of about 90 MB each, then embeds the corpus once, which is most of the wall time. On a laptop this cell took 5 to 6 minutes on two CPU threads and about 3 minutes on its GPU. It ends with the isolation check described in step 5.

In [ ]:
!lasagna benchmark experiment sweep.yaml 2>&1 | tee sweep.log

## 4. Read the result

The leaderboard ranks the arms by NDCG@10, the standard BEIR metric, over the same 100 queries.

In [ ]:
!lasagna results leaderboard results/results.json --metric ndcg@10

A leaderboard cannot say whether a gap is noise. The paired sign-flip test compares every pair of arms query by query and corrects for testing three pairs at once (Holm). Read each row as the first arm minus the second: `diff` is the change in NDCG@10, and the gap counts as real at the 5% level only when the row says `significant`. With fewer queries the same gaps can fail the test, which is why the sweep uses 100.

In [ ]:
!lasagna results significance results/results.json --metric ndcg@10 --all

## 5. Check the isolation, then publish a result others can check

The sweep's last line is the isolation check. It compares the configuration that actually reached each of the eight stages in every arm, so a setting that leaked into one arm would be named here instead of quietly moving its score.

In [ ]:
!grep -A1 "Stage isolation" sweep.log

To publish the number, bundle the run. The bundle carries the library versions, hardware, corpus fingerprint and every per-query score, and `results validate` recomputes the run id and every aggregate from those scores, so a number edited after the run no longer matches its bundle. On this run it prints VALID with no warnings: only retrieval ran and nothing in it samples, so one run per arm is the whole answer, and the run recorded the dataset revision it loaded. A warning, when one appears, tells a reader what a run did not control: `SINGLE_RUN` when something in it samples (an LLM stage) and it ran once, `UNPINNED_DATASET` when no dataset revision was recorded.

In [ ]:
!lasagna results bundle results/results.json -o run.json
!lasagna results validate run.json

## 6. Next steps

### Does it compose with another technique?

Two techniques that each help alone can overlap once combined. A 2x2 factorial measures all four combinations on the same queries and tests the interaction. This one asks whether your reranker composes with hybrid search. It reuses the corpus embedded above, so it is quicker than the sweep.

In [ ]:
%%writefile compose.yaml
name: does keyword_overlap compose with hybrid search
benchmarks:
  - name: beir/scifact
    n_samples: 100
tracks: [retrieval]
output_dir: compose_results
pipeline:
  base_profile: speed
  factorial:
    factor_a: enable_hybrid_search
    factor_b: enable_reranking
    held_fixed: {reranker: "my_ideas:keyword_overlap", chunker: fixed, chunk_size: 2000}

Run all four combinations, then test the interaction: negative means the two techniques overlap, positive means they compound, and the interval says how sure that is.

In [ ]:
!lasagna benchmark experiment compose.yaml
!lasagna results interaction compose_results/results.json --metric ndcg@10

### Swap any of the eight stages

The same sweep file swaps every stage but the document processor, with one key in each `config_overrides` entry. Your own version of a stage is a decorated function too, or a subclass of the stage's interface.

| Stage | Key | Built-ins to try | Write your own |
|---|---|---|---|
| retriever | `retriever` | `bm25`, `splade`, `hybrid_convex` | `@lasagna.retriever` |
| reranker | `reranker` | `monot5`, `colbert`, `mmr` | `@lasagna.reranker` |
| chunker | `chunker` | `recursive`, `sentence_window`, `semantic` | `@lasagna.chunker` |
| embedder | `embedder` | `sentence_transformers`, `bge_m3` | `@lasagna.embedder` |
| query enhancer | `query_enhancer` | `hyde`, `query2doc`, `multi_query` | `@lasagna.query_enhancer` |
| agentic pipeline | `agentic_pipeline` | `ircot`, `self_ask`, `flare` | subclass `BaseAgenticPipeline` |
| generator | `generator` | `cited`, `chain_of_note` | `@lasagna.generator` |

Query enhancers, agentic loops and generators call an LLM, so they need an API key; answers are scored with `tracks: [generation]`. Document processors do not sweep, because benchmark corpora arrive as text, so check one on your own files.

### Read more

- [The guide](https://github.com/dolev31/lasagna/blob/main/docs/guide.md): write a component, compare it against a baseline, and measure it on any of the 31 benchmarks.
- [Contributing a component](https://github.com/dolev31/lasagna/blob/main/docs/contributing-components.md), and the [plugin template](https://github.com/dolev31/lasagna/tree/main/examples/lasagna-plugin-template) for publishing yours as a package.
- What the [metrics](https://github.com/dolev31/lasagna/blob/main/src/lasagna/knowledge_evaluation_sdk/docs/metrics.md) mean, and which [benchmarks](https://github.com/dolev31/lasagna/blob/main/src/lasagna/knowledge_evaluation_sdk/docs/benchmarks.md) are available.
- [The results hub](https://github.com/dolev31/lasagna/tree/main/hub), for sharing bundles on a public leaderboard.
- [The paper](https://arxiv.org/abs/2610.29722) behind Lasagna, which asks whether gains measured one component at a time survive when the components are combined.